# 01 — Dataset Exploration (MVTec LOCO AD)

**LineSight — Visual Detection of Product Assembly Errors**  
Team: Sow Achta Demba, David Gwodog

This notebook covers the **Dataset & Exploration** deliverable (15% of the grade):
visualize samples per class, count images per split, and check class balance.

It runs both locally and on Google Colab. It only needs `numpy`, `Pillow`,
and `matplotlib` — no GPU, no training.

## 0. Where is the dataset?

Set `DATA_ROOT` to the folder that **contains the category folders**
(`breakfast_box`, `juice_bottle`, `pushpins`, `screw_bag`, `splicing_connectors`).

After extracting `mvtec_loco_anomaly_detection.tar`, the layout is:
```
<DATA_ROOT>/screw_bag/train/good/000.png ...
<DATA_ROOT>/screw_bag/validation/good/...
<DATA_ROOT>/screw_bag/test/{good,logical_anomalies,structural_anomalies}/...
```

In [ ]:
from pathlib import Path

# EDIT THIS to point at your extracted dataset folder:
DATA_ROOT = Path('../data/mvtec_loco')   # local clone of the repo
# DATA_ROOT = Path('/content/mvtec_loco') # typical Colab path

CATEGORY = 'screw_bag'   # try also: juice_bottle, pushpins, breakfast_box, splicing_connectors
root = DATA_ROOT / CATEGORY
assert root.exists(), f'Not found: {root}. Fix DATA_ROOT / extract the dataset first.'
print('Exploring:', root)

## 1. Count images per split and class

MVTec LOCO is an **anomaly-detection** dataset: `train` and `validation` contain
*good* images only; defects appear only in `test/` as **logical** or **structural**
anomalies. We count each bucket.

In [ ]:
IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def count_images(folder):
    if not folder.exists():
        return 0
    return sum(1 for p in folder.rglob('*') if p.suffix.lower() in IMG_EXTS)

buckets = {
    'train/good':                 root / 'train' / 'good',
    'validation/good':            root / 'validation' / 'good',
    'test/good':                  root / 'test' / 'good',
    'test/logical_anomalies':     root / 'test' / 'logical_anomalies',
    'test/structural_anomalies':  root / 'test' / 'structural_anomalies',
}
counts = {name: count_images(path) for name, path in buckets.items()}
for name, n in counts.items():
    print(f'{name:30s} {n:4d} images')
print('-' * 40)
print(f'{"TOTAL":30s} {sum(counts.values()):4d} images')

## 2. Class-balance visualization

The grader explicitly looks for a **class-balance analysis**. Note the imbalance:
many good images, fewer (and split) defect images — exactly why we use anomaly
detection (train on good only) instead of ordinary classification.

In [ ]:
import matplotlib.pyplot as plt

names = list(counts.keys())
values = [counts[n] for n in names]
colors = ['#4C9F70' if 'good' in n else '#C44E52' for n in names]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(names, values, color=colors)
ax.set_ylabel('number of images')
ax.set_title(f'Class balance — {CATEGORY}')
ax.set_xticklabels(names, rotation=30, ha='right')
for i, v in enumerate(values):
    ax.text(i, v + 0.5, str(v), ha='center')
plt.tight_layout()
plt.show()

good_total = counts['train/good'] + counts['validation/good'] + counts['test/good']
defect_total = counts['test/logical_anomalies'] + counts['test/structural_anomalies']
print(f'Good images : {good_total}')
print(f'Defect images: {defect_total}')
if defect_total:
    print(f'Good:Defect ratio = {good_total/defect_total:.1f} : 1')

## 3. Visualize sample images from each class

A quick look at one good part, one logical anomaly, and one structural anomaly.

In [ ]:
from PIL import Image
import numpy as np

def first_image(folder):
    if not folder.exists():
        return None
    for p in sorted(folder.rglob('*')):
        if p.suffix.lower() in IMG_EXTS:
            return p
    return None

samples = {
    'good (train)':           first_image(root / 'train' / 'good'),
    'logical anomaly':        first_image(root / 'test' / 'logical_anomalies'),
    'structural anomaly':     first_image(root / 'test' / 'structural_anomalies'),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, (title, path) in zip(axes, samples.items()):
    if path is not None:
        ax.imshow(Image.open(path).convert('RGB'))
        ax.set_title(f'{title}\n{path.name}')
    else:
        ax.set_title(f'{title}\n(none found)')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Image properties (size, channels)

Confirms the resolution we resize from during training (LOCO images are large,
~1MP+, so we downscale to 256×256 for the autoencoder).

In [ ]:
sample_path = first_image(root / 'train' / 'good')
if sample_path:
    img = Image.open(sample_path).convert('RGB')
    arr = np.asarray(img)
    print('Example image :', sample_path.name)
    print('Size (W x H)  :', img.size)
    print('Array shape   :', arr.shape)
    print('Dtype / range :', arr.dtype, f'[{arr.min()}, {arr.max()}]')
    print('Mean RGB      :', arr.reshape(-1, 3).mean(axis=0).round(1))

## 5. Takeaways for the report

- Dataset is **imbalanced by design** → anomaly detection (train on good only).
- Two defect families: **logical** (global/contextual) vs **structural** (local) —
  our `DecisionAgent` distinguishes them via the heatmap hot-fraction.
- Images are high-resolution → resized to 256×256 for the autoencoder.

Next: `02_train_and_evaluate_colab.ipynb` to fine-tune and evaluate a model.